# 05 增益模型与预算模拟

预测谁会买与预测谁因邮件而增量购买不同。模型参数固定，所有策略选择只在验证集进行。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src.data import download
from src.config import OUT
df = download()
print(f"客户数：{len(df):,}；字段数：{len(df.columns)}")

客户数：64,000；字段数：12


## 一、S/T/X-Learner
S 使用一个含处理变量的响应模型；T 分别拟合两组；X 通过交叉拟合响应模型生成伪效应。复用 src/modeling.py，避免第二套分析逻辑。

In [2]:
from src.modeling import run_models
from src.budget import budget_analysis
if not (OUT / 'uplift_metrics.csv').exists():
    run_models(df)
if not (OUT / 'budget_locked_policies.csv').exists():
    budget_analysis()
metrics = pd.read_csv(OUT / 'uplift_metrics.csv')
print(metrics[(metrics.outcome == 'conversion') & (metrics.split == 'test')].to_string(index=False))

   outcome split               model  auuc_raw_per_customer  auuc_above_random_per_customer  qini_normalized_sklift  uplift_auc_normalized_sklift  uplift_at_10  uplift_at_30
conversion  test Response prediction               0.002114                       -0.000047               -0.005102                     -0.000214      0.010198      0.006163
conversion  test           S-Learner               0.002226                        0.000065                0.012797                      0.000603      0.008105      0.004000
conversion  test           T-Learner               0.002657                        0.000496                0.067640                      0.003024      0.010981      0.006238
conversion  test           X-Learner               0.002521                        0.000360                0.045018                      0.001961      0.006577      0.005060


## 二、为什么不用准确率或 ROC AUC 作为增益标准
它们评价响应预测，不评价干预的增量。Qini、AUUC、Uplift@K 和分位数组检查因果增益排序。

In [3]:
print(pd.read_csv(OUT / 'prediction_metrics.csv').to_string(index=False))
print(pd.read_csv(OUT / 'uplift_deciles.csv').query("outcome == 'conversion' and model == 'S-Learner'").to_string(index=False))

   outcome      split  roc_auc   pr_auc  prevalence  brier_score
conversion validation 0.626287 0.014464    0.009297     0.009195
conversion       test 0.600896 0.014495    0.008281     0.008202
     visit validation 0.635169 0.214652    0.143437     0.119549
     visit       test 0.621519 0.215969    0.149219     0.124071
   outcome     model  decile  n_t  n_c  events_t  events_c  mean_predicted_score  observed_uplift    ci_low  ci_high
conversion S-Learner       1  855  425        19         6              0.011071         0.008105 -0.010064 0.022516
conversion S-Learner       2  852  428        10         3              0.007146         0.004728 -0.009695 0.015502
conversion S-Learner       3  846  434         7         4              0.005339        -0.000942 -0.015805 0.009424
conversion S-Learner       4  847  433         2         1              0.004355         0.000052 -0.010740 0.006544
conversion S-Learner       5  834  446         9         1              0.003668         0

## 三、验证集锁定策略
低成本不一定要求缩减覆盖。不得看到测试集 T 更好，就改选 T 并声称策略得到了独立验证。

In [4]:
import json
print(json.loads((OUT / 'policy_lock.json').read_text()))
print(pd.read_csv(OUT / 'locked_policy_test.csv').to_string(index=False))

{'selected_model': 'S-Learner', 'selected_fraction': 1.0, 'selected_on': 'validation only; no test tuning', 'selection_rule': 'uplift model with largest validation AUUC above random, then fraction with largest validation profit', 'cost_usd_per_email': 0.02, 'assumed_margin_usd_per_incremental_purchase': 30.0, 'mixture_policy': 'Any email retains randomized Mens/Womens mixture; does not select email content.', 'features': ['recency', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel'], 'seed': 20261011, 'hyperparameters_fixed': True}
                         policy  fraction  incremental_conversions_per_10000  net_profit_per_10000  bootstrap_profit_ci_low  bootstrap_profit_ci_high
                Selected policy       1.0                          43.223138           1096.694139               147.541131               2045.987315
    Random expected same budget       1.0                          43.223138           1096.694139               147.541131               2045.987315
R

## 四、预算上限、收益差和敏感性
假设贡献利润和邮件成本不是实际测得的数据。收益差区间跨零时不能证明模型优于随机投放。

In [5]:
print(pd.read_csv(OUT / 'budget_locked_policies.csv').to_string(index=False))
print(pd.read_csv(OUT / 'budget_policy_contrasts.csv').to_string(index=False))
print(pd.read_csv(OUT / 'sensitivity_analysis.csv').to_string(index=False))

 budget_cap  selected_fraction     model            strategy  incremental_conversions_per_10000  email_cost_per_10000  net_profit_per_10000  profit_ci_low  profit_ci_high
        0.1                0.1 S-Learner            Selected                           8.169647                  20.0            225.089423    -232.342879      682.237884
        0.1                0.1 S-Learner     Random expected                           4.322314                  20.0            109.669414      14.754113      204.598732
        0.1                0.1 S-Learner Response prediction                          10.521187                  20.0            295.635600    -126.243276      682.237884
        0.3                0.3 S-Learner            Selected                          11.650233                  60.0            289.507000    -378.999654      922.747574
        0.3                0.3 S-Learner     Random expected                          12.966941                  60.0            329.008242      

## 五、推荐阅读顺序
查看 reports/analysis.md 中的 Qini、分位数组、预算图和局限性，再阅读 docs/04-interview-guide.md。图表是 src/reporting.py 从结果表生成的，不手工填数字。